In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

In [2]:
from google.colab import drive
from google.colab import files
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data_dir = '/content/drive/MyDrive/Pollen Training Images'
csv_file = '/content/image_counts.csv'

df = pd.read_csv(csv_file).dropna()
df['filename'] = df['filename'].apply(lambda x: os.path.join(data_dir, x))

In [4]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Create ImageDataGenerator instances
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filename',
    y_col='count',
    target_size=(150, 150),
    batch_size=10,
    class_mode='raw',
    seed=42,
    subset='training'
)

validation_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filename',
    y_col='count',
    target_size=(150, 150),
    batch_size=10,
    class_mode='raw',
    seed=42
)


print("Train Generator Samples:", train_generator.samples)
print("Validation Generator Samples:", validation_generator.samples)


Found 288 validated image filenames.
Found 72 validated image filenames.
Train Generator Samples: 288
Validation Generator Samples: 72


In [5]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(512, activation='relu'),
    Dense(1)  # No activation function for regression
])

model.compile(
    loss='mse',  # Use mean squared error loss for regression
    optimizer='adam'
)

# Step 5: Training
history = model.fit(
    train_generator,
    steps_per_epoch=10,
    epochs=20,
    validation_data=validation_generator,
    validation_steps=50
)

Epoch 1/20
10/10 [==============================] - ETA: 0s - loss: 1114.4319

10/10 [==============================] - 179s 19s/step - loss: 1114.4319 - val_loss: 786.6624
Epoch 2/20
10/10 [==============================] - 53s 5s/step - loss: 975.2617
Epoch 3/20
10/10 [==============================] - 37s 4s/step - loss: 639.2119
Epoch 4/20
10/10 [==============================] - 26s 2s/step - loss: 456.9300
Epoch 5/20
10/10 [==============================] - 20s 2s/step - loss: 251.3308
Epoch 6/20
10/10 [==============================] - 16s 2s/step - loss: 168.2916
Epoch 7/20
10/10 [==============================] - 15s 1s/step - loss: 96.8807
Epoch 8/20
10/10 [==============================] - 13s 1s/step - loss: 157.0686
Epoch 9/20
10/10 [==============================] - 14s 1s/step - loss: 172.9753
Epoch 10/20
10/10 [==============================] - 13s 1s/step - loss: 252.6757
Epoch 11/20
10/10 [==============================] - 12s 1s/step - loss: 197.9187
Epoch 12/20
10/10 [==============================] - 12s 1s/step - loss: 528.9404
Epoch 13/20
1

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filename',
    y_col='count',
    target_size=(150, 150),
    batch_size=20,
    class_mode='raw',  # Use 'raw' to return numpy arrays
    seed=42
)

test_loss = model.evaluate(test_generator)
print('Test loss (MSE):', test_loss)

Found 72 validated image filenames.
4/4 [==============================] - 7s 2s/step - loss: 51.5256
Test loss (MSE): 51.52561569213867


In [ ]:
model.save("/content/trained_model.h5")

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
!pip install tensorflowjs

In [ ]:
!tensorflowjs_converter --input_format=keras /content/trained_model.h5 tfjs_model

In [ ]:
from google.colab import files
!zip -r tfjs_model.zip tfjs_model
files.download('tfjs_model.zip')